# ДЗ1. CLAP. Обучение проекции из аудио в текстовое пространство CLIP

**Описание задания**

В этом задании вы построите упрощенный вариант модели CLAP (Contrastive Language-Audio Pretraining):

- аудио прогоняется через предобученный аудио-энкодер (например, `LanguageBindAudio`, `PANNs CNN14` или другой)
- текстовое описание пропускается через предобученный текстовый энкодер CLIP
- поверх аудио-векторов обучается линейный адаптер, который отображает аудио в то же пространство, что и текстовые эмбеддинги CLIP
- обучение идёт по *контрастивному лоссу*, все энкодеры заморожены
- качество полученного аудио-текстового пространства оценивается на задаче поиска текстовых описаний по аудио на `AudioCaps`.

Идея оценки: хотим, чтобы для аудио и его описания косинусное сходство эмбеддингов было выше, чем для аудио и нерелевантных текстов.


**Формулировка задач**

0. Выбор аудио-энкодера.
   Выберите и обоснуйте предобученный аудио-эмбеддер:  
   - `LanguageBindAudio`,  
   - или CNN-модель (например, PANNs CNN14),  
   - или другой открытый аудио-энкодер, который выдаёт фиксированный эмбеддинг.

1. Подсчёт эмбеддингов.
   - Посчитайте аудио-векторы для всех аудио из `AudioCaps` с помощью выбранного энкодера.  
   - Посчитайте текстовые векторы для подписей с помощью `CLIP text encoder`.

2. Линейная аудио-проекция.
   - Реализуйте модель `AudioProjection`, переводящую аудио-эмбеддинг в размерность текстового эмбеддинга CLIP.

3. Контрастивное обучение.
   - Обучите аудио-проекцию на датасете `AudioCaps` по схеме аудио ↔ текст с контрастивным лоссом.  
   - Аудио-энкодер и CLIP должны быть полностью заморожены.

4. Оценка качества.
   - Оцените качество полученного аудио-текстового пространства на задаче поиска текстовых описаний по аудио:  
     для каждого аудио найдите наиболее похожие текстовые подписи в выбранном для оценки разбиении и посчитайте `R@1/3/10`.  
   - Сравните результаты до и после обучения, а также со *случайным бейзлайном*.


### Сеттинг

> Подготовьте все необходимые импорты и загрузите необходимые данные.

Для загрузки AudioCaps воспользуемся [MP3-версией на Hugging Face](https://huggingface.co/datasets/JackyHoCL/AudioCaps-mp3).

В колонке `audio` хранятся байты MP3-файлов. Декодирование и подготовку аудио организуйте под выбранный энкодер.

In [ ]:
%pip install -q "datasets==4.5.0" "soundfile>=0.13"

In [ ]:
# для загрузки audiocaps можно воспользоваться этим кодом
from datasets import Audio, load_dataset

audiocaps = load_dataset(
    "JackyHoCL/AudioCaps-mp3",
    revision="3f1766c37bea96f49538b11b790162129274bf8c",
)

# оставляем аудио в виде байтов для обработки выбранной моделью
audiocaps = audiocaps.cast_column("audio", Audio(decode=False))
audiocaps

### Задание 1. Подготовка аудио- и текстовых энкодеров (2 балла)

В этом задании вам нужно:

1. Выбрать аудио-энкодер и инициализировать его.
2. Инициализировать текстовый энкодер CLIP. Вы свободны выбирать самостоятельно, какой именно.
3. Заморозить параметры обоих энкодеров (мы не дообучаем их, а учим только линейный адаптер).

Вы можете:

* использовать `LanguageBindAudio`
* или подставить свою аудио-модель (главное - чтобы на выходе был вектор фиксированной размерности).

Обоснуйте выбор моделей и укажите, какие веса используете. Опишите, на каких данных обучался аудио-энкодер. Для текста используйте финальные эмбеддинги из общего пространства CLIP, после проекционной головы.


In [ ]:
# your code here
# ┌(ಠ_ಠ)┘

### Задание 2. Предподсчёт аудио- и текстовых эмбеддингов (3 балла)

> Важный момент, который пригодится вам и в других домашних.

Чтобы не тратить время на многократный прогон энкодеров при обучении, следует:

1. Предварительно посчитать аудио-эмбеддинги для каждой записи из `AudioCaps`.
2. Записывать их в файл формата `pickle` (например), где ключ - идентификатор аудиозаписи, значение - numpy-вектор.
3. Аналогично посчитать текстовые эмбеддинги для подписей через CLIP и совместить их с аудио.

Рекомендуемая структура:

* функция `extract_audio_vectors_with_checkpointing(...)` - обходит аудиозаписи, считает эмбеддинги, периодически делает чекпоинты
* функция `extract_text_embeddings(texts, clip_model, clip_processor)` - возвращает список текстовых эмбеддингов
* функция `process_dataset(...)` - читает данные AudioCaps, мержит аудио-эмбеддинги и текстовые, сохраняет список словарей вида  
  `{"uniq_id": ..., "audio_embedding": ..., "text_embedding": ...}` в pickle.

> Вы вольны отходить от предлагаемой структуры.

In [ ]:
# your code here (づ｡◕‿‿◕｡)づ

### Задание 3. Линейный аудио-адаптер и контрастивный лосс (3 балла)


Теперь, когда у нас есть пары *audio_embedding, text_embedding*, реализуем:

1. Подготовку батчей из сохранённых эмбеддингов. Можно написать класс `AudioTextDataset` или воспользоваться готовыми средствами PyTorch.
2. Линейную модель `AudioProjection`, переводящую аудио-эмбеддинг в размерность текстового.
3. Симметричный контрастивный лосс для аудио↔текст.

Обучаем `AudioProjection`, энкодеры оставляем замороженными. Сохраните веса проекции до и после обучения и постройте график лосса.

> Обратите внимание на масштаб значений в матрице сходства и на то, какие данные влияют на выбор параметров.


In [ ]:
# your code here
# (╯°□°）╯︵ ┻━┻

### Задание 4. Оценка качества на задаче поиска текстовых описаний по аудио (2 балла)


Теперь нужно понять, насколько хорошо аудио-векторы после проекции "попадают" в пространство текстовых эмбеддингов:

1. Посчитайте проекции аудио для всех уникальных аудиозаписей в выбранном для оценки разбиении.
2. Для каждого аудио найдите `top-k` наиболее похожих текстов среди всех подписей этого разбиения по косинусному сходству (или скалярному произведению после L2-нормализации).
3. Посчитайте `R@1`, `R@3`, `R@10`, т.е. долю случаев, когда хотя бы одна "правильная" подпись попала в топ-k.
4. Сравните с неким *случайным бейзлайном*: для каждого аудио выберите `k` случайных текстов и посчитайте такую же метрику.
5. Сравните результаты до и после обучения на одних и тех же запросах и подписях. Укажите разбиение, число запросов и размер набора подписей.
6. Посмотрите несколько удачных и неудачных примеров и разберите ошибки.

> У одной аудиозаписи может быть несколько подписей. Все они считаются правильными. Аудиозапись определяется парой `(youtube_id, start_time)`.

In [ ]:
# your code here
# (⌐■_■)

### Бонус. Поиск изображений по звуку (2 балла)

У CLIP есть ещё и визуальный энкодер. Давайте проверим, можно ли использовать обученную аудио-проекцию для поиска изображений по звуку.

Для этого возьмём VGGSound. В этой версии есть 15 446 записей с аудио, видео и метками 309 звуковых классов: [аудио и видео](https://huggingface.co/datasets/11hu83/vggsound), [метки классов](https://huggingface.co/datasets/lenghanz/VGGSound/blob/main/test.csv).

Правильными ответами будем считать клипы с тем же звуковым классом, что и аудиозапрос. Таких ответов в галерее может быть несколько, включая исходный клип.

В этом задании вам нужно:

1. Выбрать аудиозапросы и подготовить галерею кадров. Объём выборки и способ отбора записей выберите самостоятельно и обоснуйте. Укажите число запросов, кандидатов и классов.
2. Посчитать визуальные эмбеддинги с помощью той же модели CLIP с теми же весами, что использовали для текста. Нужны финальные эмбеддинги из общего пространства, после проекционной головы. Можно выбрать один кадр или объединить признаки нескольких кадров. При этом один клип должен оставаться одним кандидатом в галерее.
3. Использовать аудио-проекцию из основной части без дополнительного обучения на парах аудио-изображение. Сравнивать аудио- и визуальные эмбеддинги напрямую. Метки классов используйте для оценки, а не как вход модели при поиске.
4. Посчитать `R@1`, `R@3`, `R@10` по всей выбранной галерее: долю запросов, для которых в топ-k нашёлся хотя бы один клип того же класса. Для каждого запроса в галерее должен быть правильный ответ. Сравните результаты до и после обучения проекции и со случайным бейзлайном на одних и тех же запросах и кандидатах.
5. Покажите несколько результатов вместе со звуком и изображениями и разберите ошибки. Учтите ограничения разметки и то, виден ли источник звука в выбранных кадрах. Сопоставьте результат с основной задачей и предобучением аудио-энкодера.

> При выборе записей обратите внимание на происхождение данных в основной части и бонусе.

За эксперимент - 1 балл, за сравнения и разбор результатов - ещё 1 балл. Улучшение метрик не обязательно, но полученный результат нужно проверить и объяснить.

Про связывание модальностей можно почитать в [ImageBind](https://arxiv.org/abs/2305.05665) и [LanguageBind](https://arxiv.org/abs/2310.01852).

Код ниже загружает только таблицу с метками и ссылками. Аудио и видео скачайте для выбранных вами записей.


In [ ]:
import pandas as pd

labels_url = (
    "https://huggingface.co/datasets/lenghanz/VGGSound/resolve/"
    "467dacfcdcecb4156a3b96c6fee5d0aa0ce8b039/test.csv"
)
vggsound = pd.read_csv(labels_url, header=None, names=["filename", "label"])
vggsound["video_id"] = vggsound["filename"].str.removesuffix(".mp4")

# ссылки ведут на отдельные записи, поэтому можно скачать только свою выборку
media_root = (
    "https://huggingface.co/datasets/11hu83/vggsound/resolve/"
    "dc7815aa65132c48f070686e0004180f03b5abcf/"
)
vggsound["audio_url"] = media_root + "audio/" + vggsound["video_id"] + "/audio.wav"
vggsound["video_url"] = media_root + "video/" + vggsound["video_id"] + "/video.mp4"
vggsound.head()

In [ ]:
# your code here
# ᕕ( ᐛ )ᕗ

### Вывод

Оформите, пожалуйста, небольшой вывод. Например, можно воспользоваться следующим планом:

   * какую аудио-модель вы выбрали и почему
   * как вели себя потери на обучении
   * какие значения метрик получились до и после обучения и как они соотносятся со случайным baseline
   * любые наблюдения (например, зависимость от числа эпох, размера батча и т.д.)
   * если делали бонус, что получилось с поиском изображений по звуку
   * милые пожелания ассистенту/лектору, который будет это проверять.

your text here (ಠ.ಠ)